In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from nltk.corpus import stopwords
import nltk
import sqlite3

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

db_path = '/Users/zphilipp/git/research/dealsdb/deals_db1.db'

prepositions_and_conjunctions = [
    "about", "above", "across", "after", "against", "along", "among", "around", "at",
    "before", "behind", "below", "beneath", "beside", "between", "beyond", "by",
    "during", "for", "from", "in", "inside", "into", "near", "of", "off", "on",
    "out", "outside", "over", "through", "throughout", "to", "toward", "under",
    "until", "up", "with", "within", "without", "and", "but", "or", "for", "nor",
    "so", "yet", "although", "because", "as", "since", "unless", "while", "when",
    "where", "after", "before", "the", "a", "b", "c", "d", "e", "f", "g", "h",
    "i", "j", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w",
    "x", "y", "z"
]
pattern = r'\b(?:' + '|'.join(prepositions_and_conjunctions) + r')\b'

def remove_prepositions_and_conjunctions(text):
    cleaned_text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r'\d+', '', cleaned_text)
    return re.sub(r'\s+', ' ', cleaned_text).strip()

nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/zphilipp/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

sql_query = """
    SELECT d.id, c.name || '. ' ||  GROUP_CONCAT(d.title_general) || '. '||GROUP_CONCAT (o.title) as text
    FROM deals d
        JOIN customer_category c ON c.id=d.customer_category_id
        JOIN options o ON o.deal_id=d.id
    GROUP BY d.id
"""

# Execute the query and load the data into a DataFrame
df = pd.read_sql_query(sql_query, conn)
conn.close()

In [5]:
df.head()

,id,text
0,1,"Barber Shop. Transform Your Look with Men's Haircut with Optional Wash and Beard Trim at 028 Barber School (Up to 64% Off),Transform Your Look with Men's Haircut with Optional Wash and Beard Trim at 028 Barber School (Up to 64% Off),Transform Your Look with Men's Haircut with Optional Wash and Beard Trim at 028 Barber School (Up to 64% Off). One Men's Haircut, Beard trim and Wash,One Men's Haircuts and Wash,One Men's Haircut"
1,2,"Things To Do. Experience Jersey Axe House with Admission for Groups of 4, 6, or 8 People and Save up to 22%,Experience Jersey Axe House with Admission for Groups of 4, 6, or 8 People and Save up to 22%. 2-Hour Axe Throwing Experience for 8 People (Ages 12 and Up),2-Hour Axe Throwing Experience for 6 People (Ages 12 and Up)"
2,3,"Junk Removal. Experience efficient junk removal with 1-800-Junk-Refund, hauling away 1/4 truck load of items, up to 56% off.. One Quarter Truck Load of Junk Removal"
3,4,Junk Removal. Clear Out Clutter With Our 1/4 Truck Load Junk Removal Service from 1-800-Junk-Refund (Up to 52% Off). 1/4 Truck Load of Junk Removal
4,5,"Bars. Explore the Country Bar Crawl with Las Vegas Tours featuring a Party Bus, Mixed Drinks, and VIP Access up to 58% off,Explore the Country Bar Crawl with Las Vegas Tours featuring a Party Bus, Mixed Drinks, and VIP Access up to 58% off,Explore the Country Bar Crawl with Las Vegas Tours featuring a Party Bus, Mixed Drinks, and VIP Access up to 58% off,Explore the Country Bar Crawl with Las Vegas Tours featuring a Party Bus, Mixed Drinks, and VIP Access up to 58% off. Admit up to EIGHT: #1 Country Bar Crawl in Vegas w/ Party Bus & Mixed Drinks,Admit FOUR: #1 Country Bar Crawl in Vegas w/ Party Bus & Mixed Drinks,Admit TWO: #1 Country Bar Crawl in Vegas w/ Party Bus & Mixed Drinks,Admit ONE: #1 Country Bar Crawl in Vegas w/ Party Bus & Mixed Drinks"


In [6]:
tfidf_vectorizer = TfidfVectorizer(stop_words=stopwords.words('english'))
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'])

number_of_clusters = 100
kmeans = KMeans(n_clusters=number_of_clusters)
kmeans.fit(tfidf_matrix)


KMeans(n_clusters=100)

In [7]:
df['cluster'] = kmeans.labels_

In [14]:
df[['text', 'cluster']].query("cluster == 1").head(10)

,text,cluster
118,"Facial. One or Three Hydrodermabrasion Sessions at 2 Generations Beauty And Spa (Up to 50% Off),One or Three Hydrodermabrasion Sessions at 2 Generations Beauty And Spa (Up to 50% Off). One Hydrodermabrasion Session,Three Hydrodermabrasion Sessions",1
185,"Colonic Hydrotherapy. Up to 47% Off on Colonic / Hydro Colon Therapy at 247 Natural Wellness Center,Up to 47% Off on Colonic / Hydro Colon Therapy at 247 Natural Wellness Center. One Colon-Hydrotherapy (Colonic) Treatment for Returning Clients,One Colon-Hydrotherapy Treatment",1
251,"Natural Medicine. Experience advanced healing at 34th Street Chiropractic with varied Hyperbaric Oxygen Therapy sessions, offering up to 30% off. One 60-Minute Hyperbaric Oxygen Therapy Session",1
252,Spas. Discover 34th Street Chiropractic And Wellness' red light therapy sessions with up to 37% off. Red Light Therapy,1
255,"Weight Loss. Experience 360 Tan's Infrared Therapy Sessions for pain relief and skin health, offering up to 28% off without appointments.. Ten Infrared Red Light Therapy Sessions",1
263,"Facial. Experience rejuvenating facial oxygen therapy at 360 Radiance, offering advanced ultrasonic exfoliation and vacuum therapy up to 43% off. Oxygen Facial Therapy with Ultrasonic Exfoliation, Vacuum Therapy, and Customized Nutritional Serum",1
268,"Spas. Revitalize with One or Two Full Body Red Light Therapy Sessions at 360 Tans (Up to 74% Off),Revitalize with One or Two Full Body Red Light Therapy Sessions at 360 Tans (Up to 74% Off). One Full Body Red Light Therapy Session,Two Full Body Red Light Therapy Session",1
330,"Salt Caves. Experience the ultimate relaxation at 4 Elements Wellness Center with a 60-minute salt cave session for one or two, up to 46% off,Experience the ultimate relaxation at 4 Elements Wellness Center with a 60-minute salt cave session for one or two, up to 46% off. One 60-Minute Himalayan Salt Room Session for One,One 60-Minute Himalayan Salt Room Session for Two",1
336,"Medical. One or Three Energy & Metabolism Boosting Vitamin IV drip at 4Ever Young Doral(Up To 52% Off),One or Three Energy & Metabolism Boosting Vitamin IV drip at 4Ever Young Doral(Up To 52% Off). Three Energy & Metabolism Boosting Vitamin IV drip,One Energy & Metabolism Boosting Vitamin IV drip",1
338,"Medical. One or Three Energy & Metabolism Boosting Vitamin IV Drips at 4Ever Young Fleming Island (Up to 52% Off),One or Three Energy & Metabolism Boosting Vitamin IV Drips at 4Ever Young Fleming Island (Up to 52% Off). One Energy & Metabolism Boosting Vitamin IV Drip,Three Energy & Metabolism Boosting Vitamin IV Drip",1


In [15]:
df[['text', 'cluster']].query("cluster == 2").head(10)

,text,cluster
79,"Massage. Up to 51% Off on Lymphatic Drainage Massage at 124 Wellness Studio,Up to 51% Off on Lymphatic Drainage Massage at 124 Wellness Studio. Three 60-Minute Compression Lymphatic Drainage Massages for Legs & Massage,One 30-min Compression Lymphatic Drainage Massage For Legs Or Hips",2
103,"Weight Loss. One or Three Facial Endermologie w/ Instant Lift and Lymphatic Drainage at 1917 spa (Up to 60% Off),One or Three Facial Endermologie w/ Instant Lift and Lymphatic Drainage at 1917 spa (Up to 60% Off). 3 Facial Endermologie (instant lift and lymphatic drainage),1 Facial Endermologie (instant lift and lymphatic drainage)",2
569,"Massage. Get One or Three Lymphatic Drainage Sessions to Boost Wellness at 5 Elements Care and Solutions (Up To 60% Off),Get One or Three Lymphatic Drainage Sessions to Boost Wellness at 5 Elements Care and Solutions (Up To 60% Off). Three 45-Minute Lymphatic Drainage Sessions,One 45-Minute Lymphatic Drainage Session",2
739,"Massage. Lymphatic Drainage Therapy Sessions at A&N Beauty Bar (Up to 60% Off),Lymphatic Drainage Therapy Sessions at A&N Beauty Bar (Up to 60% Off). One Noninvasive Lymphatic Drainage Therapy Session,Three Noninvasive Lymphatic Drainage Therapy Sessions",2
1118,"Massage. 60-Minute Lymphatic Drainage Massage or Upgrade to 90-Minute Signature Option for Detoxification(Up To 55% Off),60-Minute Lymphatic Drainage Massage or Upgrade to 90-Minute Signature Option for Detoxification(Up To 55% Off). 60-Minute Lymphatic Drainage Massage,90-Minute Signature Lymphatic Drainage Massage",2
1445,"Massage. Experience tailored Brazilian lymphatic drainage massages at Abundance WellSpa with up to 54% off for enhanced recovery sessions.,Experience tailored Brazilian lymphatic drainage massages at Abundance WellSpa with up to 54% off for enhanced recovery sessions.,Experience tailored Brazilian lymphatic drainage massages at Abundance WellSpa with up to 54% off for enhanced recovery sessions.. One 45-Minute Brazilian Lymphatic Drainage Massage,6 45-Minute Brazilian Lymphatic Drainage Massage,Three 45-Minute Brazilian Lymphatic Drainage Massages",2
2115,"Massage. Discover Personalized Plans for Body Contouring with Lymphatic Drainage Treatments (Up to 90% Off)\n,Discover Personalized Plans for Body Contouring with Lymphatic Drainage Treatments (Up to 90% Off)\n,Discover Personalized Plans for Body Contouring with Lymphatic Drainage Treatments (Up to 90% Off)\n,Discover Personalized Plans for Body Contouring with Lymphatic Drainage Treatments (Up to 90% Off)\n. One Fat Loss and Lymphatic Drainage Treatment,Four Fat Loss and Lymphatic Drainage Treatments,Three Fat Loss and Lymphatic Drainage Treatments,Two Fat Loss and Lymphatic Drainage Treatments",2
2192,Massage. Up to 37% Off on Lymphatic Drainage Massage at Aesthetically You and Weight Loss Too. 30-Minute Lymphatic Drainage Treatment,2
2441,"Massage. One 45-min Lymphatic Drainage Massage for Hips, Legs,/Arms (Compression) at Ageless Wellness Spa (Up to 37% Off),One 45-min Lymphatic Drainage Massage for Hips, Legs,/Arms (Compression) at Ageless Wellness Spa (Up to 37% Off),One 45-min Lymphatic Drainage Massage for Hips, Legs,/Arms (Compression) at Ageless Wellness Spa (Up to 37% Off). One 45-Minute Lymphatic Drainage Massage for arms (Compression),One 45-Minute Lymphatic Drainage Massage for hips (Compression),One 45-Minute Lymphatic Drainage Massage for legs (Compression)",2
2477,"Massage. Unwind at Ahoy Therapy LLC with Post-Op Lymphatic and Zero Gravity Massages up to 34% off,Unwind at Ahoy Therapy LLC with Post-Op Lymphatic and Zero Gravity Massages up to 34% off. Zero Gravity Chair Full Body Massage,Post Op Lymphatic Drainage Massage",2


In [13]:
df[['text', 'cluster']].query("cluster == 3").head(10)

,text,cluster
3270,"Mini Golf. One Round of Indoor Glow Golf for 2, 4, or 6 at Aloha Mini Glow Golf - Mall of New Hampshire (Up to 31% Off),One Round of Indoor Glow Golf for 2, 4, or 6 at Aloha Mini Glow Golf - Mall of New Hampshire (Up to 31% Off),One Round of Indoor Glow Golf for 2, 4, or 6 at Aloha Mini Glow Golf - Mall of New Hampshire (Up to 31% Off). One Round of Indoor Glow Golf for Six,One Round of Indoor Glow Golf for Two,One Round of Indoor Glow Golf for Four",3
11182,"Golf. Experience Custom Fit Clubs at Brians Golf Works with Options Worth $100, Up to 50% Off. $100 Golf Club Fitting Session",3
11343,"Golf. Enjoy a Relaxing 9-Hole Round of Golf for One, Two, or Four People at Brookland Golf Course (Up to 37% Off),Enjoy a Relaxing 9-Hole Round of Golf for One, Two, or Four People at Brookland Golf Course (Up to 37% Off),Enjoy a Relaxing 9-Hole Round of Golf for One, Two, or Four People at Brookland Golf Course (Up to 37% Off). 9-Hole Round of Golf for Two People,9-Hole Round of Golf for Four People,9-Hole Round of Golf for One Person",3
11727,"Golf. Tee Off Anytime with Up to Four Hours of Indoor Golf at Bunker Hill Golf Course (Up to 38% Off),Tee Off Anytime with Up to Four Hours of Indoor Golf at Bunker Hill Golf Course (Up to 38% Off),Tee Off Anytime with Up to Four Hours of Indoor Golf at Bunker Hill Golf Course (Up to 38% Off). Two Hours of Indoor Golf For Up to 6 People,One Hour of Indoor Golf For Up to 6 People,Four Hours of Indoor Golf For Up to 6 People",3
13881,"Golf. At Centerbrook Golf Course, experience 9 holes of golf with cart for up to 20% off,At Centerbrook Golf Course, experience 9 holes of golf with cart for up to 20% off. 9-Hole Round of Golf for Two with Cart,9-Hole Round of Golf for Four with Cart",3
14248,"Golf. 9- or 18-Round of Golf w/ Cart for 2 or 4 at Cherry Valley Golf Course(Up to 47% Off),9- or 18-Round of Golf w/ Cart for 2 or 4 at Cherry Valley Golf Course(Up to 47% Off),9- or 18-Round of Golf w/ Cart for 2 or 4 at Cherry Valley Golf Course(Up to 47% Off),9- or 18-Round of Golf w/ Cart for 2 or 4 at Cherry Valley Golf Course(Up to 47% Off). 9-Hole Round of Golf with Cart for 4 (Monday-Friday Anytime, Sat/Sun after 2),18-Hole Round of Golf w/ Cart for 2 (Monday-Friday anytime, Sat/Sun after 2pm),9- Hole Round of Golf with Cart for 2 (Monday-Friday),18-Hole Round of Golf w/ Carts for 4 (Monday-Friday Anytime)",3
14533,"Golf. Relax at Chosen Valley Golf Club with walking golf for groups and individuals, plus enjoy up to 42% off.,Relax at Chosen Valley Golf Club with walking golf for groups and individuals, plus enjoy up to 42% off.,Relax at Chosen Valley Golf Club with walking golf for groups and individuals, plus enjoy up to 42% off.. 9-Hole Round of Golf for Four,9-Hole Round of Golf for Two,9-Hole Round of Golf for One",3
17002,"Golf. Discover Golf: Five-Week Learn How to Play Golf Class for One or Two at Cypresswood Golf Club (Up to 84% Off),Discover Golf: Five-Week Learn How to Play Golf Class for One or Two at Cypresswood Golf Club (Up to 84% Off). Five-Week ""Learn To Play Golf"" Class for One,Five-Week ""Learn To Play Golf"" Class for Two",3
17319,"Golf. Up to 28% Off on Golf at DAS Golf Lessons,Up to 28% Off on Golf at DAS Golf Lessons,Up to 28% Off on Golf at DAS Golf Lessons,Up to 28% Off on Golf at DAS Golf Lessons. One - 60 Minute Golf Lesson with with a copy of ""The Method: A Golf Success Strategy ”,Three - 60 Minute Golf Lessons with with a copy of “Your Yardage Book”,Two - 60 minute Golf Lessons with with a copy of “Your Yardage Book”,One - 60 minute Golf Lesson with with a copy of “Your Yardage Book”",3
21418,"Golf. Up to 44% Off on Indoor Golf at Envision Golf,Up to 44% Off on Indoor Golf at Envision Golf. One hour Standard Bay Golf Rental for up to 6 people, Valid to,Two hour Standard Bay Golf Rental for up to 6 people, Valid to",3
